## CS310 Natural Language Processing
## Lab 10: Direct Preference Optimization

In this lab, we will practice implementing the key steps of conducting direct preference optimization (DPO) for achieving better LLM human alignment.

In [19]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from functools import partial
import pprint
import json
from utils import GPTModel

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: cuda


## T1. Prepare Preference Data for DPO

We will load the preference data `instruction-data-with-preference.json`.



In [20]:
DATA_PATH = 'instruction-data-with-preference.json'

def get_data(data_path):
    with open(data_path, 'r') as f:
        text_data = f.read()    
    data = json.loads(text_data)
    return data

# Print some data
data = get_data(DATA_PATH)
pprint.pp(data[0])


{'instruction': 'Evaluate the following phrase by transforming it into the '
                'spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the '
           'correct spelling is "friend".',
 'rejected': 'The spelling of the given phrase "freind" is flat out wrong, get '
             'it together, the correct spelling is "friend".',
 'chosen': 'The spelling of the given phrase "freind" is incorrect, the '
           'correct spelling is "friend".'}


From the output, we can see severl fields:
- `instruction` and `input`: used as LLM inputs.
- `output`: response from LLM.
- `rejected`: the dispreferred response.
- `chosen`: the preferred response.

---

Next, we will format the input and response, similar to the SFT task.

In [21]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

def format_response(response):
    """
    response: str, either entry['chosen'] or entry['rejected']
    """
    return f"\n\n### Response:\n{response}"

In [4]:
print(format_input(data[0]))
print(format_response(data[0]['chosen']))
print(format_response(data[0]['rejected']))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the following phrase by transforming it into the spelling given.

### Input:
freind --> friend


### Response:
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".


### Response:
The spelling of the given phrase "freind" is flat out wrong, get it together, the correct spelling is "friend".


Next, we will create a `PreferenceDataset` class to wrap the raw data.

It is slightly different from the `InstructionDataset` class in the previous lab: instead of returning a single output sequence, we will now return a pair of responses, "chonse" and "rejected".

Hint:
- Call `format_input` to get the prompt. 
- Call `format_response` to get the chosen and rejected responses, respectively.
- Call `tokenizer.encode` to tokenize the prompt and responses.

In [22]:
class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            ### START YOUR CODE ###
            prompt = format_input(entry)
            chosen = format_response(entry['chosen'])
            rejected = format_response(entry['rejected'])
            ### END YOUR CODE ###

            chosen_full_text = prompt + '\n\n' + chosen
            rejected_full_text = prompt + '\n\n' + rejected

            ### START YOUR CODE ###
            prompt_tokens = tokenizer.encode(prompt)
            chosen_full_tokens = tokenizer.encode(chosen_full_text)
            rejected_full_tokens = tokenizer.encode(rejected_full_text)
            ### END YOUR CODE ###

            self.encoded_texts.append({
                'prompt': prompt_tokens,
                'chosen': chosen_full_tokens, 
                'rejected': rejected_full_tokens})

    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)

Along with the `PreferenceDataset` class, we will also need a new collation function to pad the sequences in a batch to the same length.

In [23]:
def custom_collate_fn(batch, 
    pad_token_id=50256,
    allowed_max_length=None,
    mask_prompt_tokens=True,
    device="cpu"):
    """
    batch: list of dictionaries, each containing 'prompt', 'chosen', and 'rejected' keys
    """
    # Initialize lists to hold batch data
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [],
        "chosen_mask": []
    }

    # Get the max length
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max(len(item[key])+1 for item in batch)
            max_length_common = max(max_length_common, current_max)
    
    # Process each item
    for item in batch:
        prompt = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt)

        for key in ["chosen", "rejected"]:
            sequence = item[key]
            ### START YOUR CODE ###
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))
            ### END YOUR CODE ###

            # Mask all padding tokens to False
            ### START YOUR CODE ###
            mask = torch.tensor([token != pad_token_id for token in padded])
            ### END YOUR CODE ###

            # Set mask for all input tokens to False
            # +2 sets the 2 newline ("\n") tokens before "### Response" to False
            if mask_prompt_tokens:
                mask[:prompt.shape[0]+2] = False
            
            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)
    
    # Convert lists to tensors
    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        # Stack all sequences into a tensor
        tensors = torch.stack(batch_data[key])
        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            tensor_stack = tensors[:, :allowed_max_length]
        batch_data[key] = tensors.to(device)
    
    return batch_data

In [7]:
# Like usual, use `partial` to create a customized function with default arguments.
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,            # Put the data directly on a GPU if available
    mask_prompt_tokens=True,  # This is optional
    allowed_max_length=1024   # The supported context length of the model
)

Now, let's put `PreferenceDataset` and `customized_collate_fn` together, into `DataLoader`.

In [8]:
# Test
tokenizer = tiktoken.get_encoding("gpt2")

example_dataset = PreferenceDataset(data[:2], tokenizer)
print(example_dataset[0].keys())

dict_keys(['prompt', 'chosen', 'rejected'])


In [9]:
# Test
example_dataloader = DataLoader(
    example_dataset, 
    batch_size=2, 
    collate_fn=customized_collate_fn,
    shuffle=False
    )

for batch in example_dataloader:
    break

print("batch.keys:", batch.keys())
print("batch['chosen'].shape:", batch['chosen'].shape)
print("batch['rejected'].shape:", batch['rejected'].shape)

# You are expected to see the following output:
# batch.keys: dict_keys(['prompt', 'chosen', 'rejected', 'rejected_mask', 'chosen_mask'])
# batch['chosen'].shape: torch.Size([2, 82])
# batch['rejected'].shape: torch.Size([2, 82])

batch.keys: dict_keys(['prompt', 'chosen', 'rejected', 'rejected_mask', 'chosen_mask'])
batch['chosen'].shape: torch.Size([2, 82])
batch['rejected'].shape: torch.Size([2, 82])


## T2. Defining the DPO Loss

Once we have the data ready, we can now start defining the DPO loss function.

The quation for DPO loss is:

$$L_{DPO}(\pi_\theta, \pi_\text{ref}) = -\mathbb{E}_{(x, y_w, y_l)\sim D} \left[ \log\sigma\left(\text{log} \frac{\pi_\theta(y_w | x)}{\pi_\text{ref}(y_w | x)} - \text{log} \frac{\pi_\theta(y_l | x)}{\pi_\text{ref}(y_l | x)}\right) \right]$$

Hint:
- $\pi_\theta$ denotes the policy model that we want to optimize; $\pi_\text{ref}$ is the reference model; at the beginning of training, they are the same.
- $\log\pi_\theta(\cdot|x)$ are passed in as arguments `model_chosen_logits` and `model_rejected_logits`, respectively.
- $\log\pi_\text{ref}(\cdot|x)$ are passed in as arguments `ref_chosen_logits` and `ref_rejected_logits`, respectively.
- The log-odds ratio $\log\frac{\pi_\theta(y_w | \mathbf{x})}{\pi_\text{ref}(y_w | \mathbf{x})}$ is computed as the difference between `model_chosen_logits` and `ref_chosen_logits`. 
- This difference can be also understood as an implicit reward, which will be returned as `reward_chosen`. It will be useful for tracking the training progress. 
- You can use the PyTorch built-in function `F.logsigmoid` to compute $\log\sigma(\cdot)$.
- The final loss returned is the mean value.


In [24]:
def compute_dpo_loss(
      model_chosen_logprobs,
      model_rejected_logprobs,
      ref_chosen_logprobs,
      ref_rejected_logprobs,
      beta=0.1,
    ):
    """
    Compute the DPO loss.
    """
    # Compute the log-odds ratio
    ### START YOUR CODE ###
    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    ref_logratios = ref_chosen_logprobs - ref_rejected_logprobs
    ### END YOUR CODE ###

    # Compute the loss
    ### START YOUR CODE ###
    loss = -F.logsigmoid(beta * (model_logratios - ref_logratios)).mean()
    ### END YOUR CODE ###

    # Optional values to track progress during training
    chosen_rewards = (model_chosen_logprobs - ref_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - ref_rejected_logprobs).detach()

    return loss, chosen_rewards, rejected_rewards

Note that the above function relies on the log probabilities, which will be computed in the following function `compute_logprobs`.

Hint:
- What it does is to compute the cross-entropy between `logits` and `targets`, where `targets` are the input tokens shifted by one. Therefore, you can use `F.cross_entropy`.

In [27]:
def compute_logprobs(logits, targets, selection_mask=None):
    """
    Compute log probabilities.

    Args:
      logits: Tensor of shape (batch_size, num_tokens, vocab_size)
      targets: Tensor of shape (batch_size, num_tokens)
      selection_mask: Tensor for shape (batch_size, num_tokens)

    Returns:
      mean_log_prob: Mean log probability excluding padding tokens.
    """
    ### START YOUR CODE ###
    # GPT logits are [batch, seq, vocab]; predict token t from positions < t.
    logits = logits[:, :-1, :].permute(0, 2, 1) # Shift by 1
    targets = targets[:, 1:] # Shift by 1
    # cross_entropy expects [batch, num_classes, ...] — move vocab to dim 1
    log_probs = F.cross_entropy(logits, targets, reduction="none")
    ### END YOUR CODE ###

    if selection_mask is not None:
        mask = selection_mask[:, 1:].clone()
        # Apply the mask to filter out padding tokens
        log_probs = log_probs * mask
        # Calculate the average log probability excluding padding tokens
        # This averages over the tokens, so the shape is (batch_size,)
        avg_log_prob = log_probs.sum(-1) / mask.sum(-1)
        return avg_log_prob
    else:
        avg_log_prob = log_probs.mean()
        return avg_log_prob

In [28]:
# Test
torch.manual_seed(0)

# Same layout as GPTModel: (batch, seq_len, vocab_size)
logits = torch.randn(2, 10, 5, requires_grad=True)
targets = torch.randint(0, 5, (2, 10))
with torch.no_grad():
    loss1 = compute_logprobs(logits, targets)

with torch.no_grad():
    logits_s = logits[:, :-1, :].permute(0, 2, 1)
    targets_s = targets[:, 1:]
    loss2 = F.cross_entropy(logits_s, targets_s)

print("loss1:", loss1)
print("loss2:", loss2)

# You are expected to see the following output:
# loss1: tensor(1.8617)
# loss2: tensor(1.8617)
# loss1 and loss2 should match (exact values depend on seed / shapes).

loss1: tensor(1.8617)
loss2: tensor(1.8617)


Given the above functions, we can now combine them for a batch of data.

Hint:
- `logits` are returned from calling the `policy_model` or `reference_model`.

In [13]:
def compute_dpo_loss_batch(batch, policy_model, reference_model, beta):
    """Compute the DPO loss on an input batch"""

    # where policy_model(batch["chosen"]) are the logits
    ### START YOUR CODE ###
    # Hint: Call `compute_logprobs` using logits produced by policy model
    policy_chosen_log_probas = compute_logprobs(policy_model(batch["chosen"]), batch["chosen"], batch["chosen_mask"])
    policy_rejected_log_probas = compute_logprobs(policy_model(batch["rejected"]), batch["rejected"], batch["rejected_mask"])
    ### END YOUR CODE ###
    
    with torch.no_grad():
        ### START YOUR CODE ###
        # Hint: Call `compute_logprobs` using logits produced by reference model
        ref_chosen_log_probas = compute_logprobs(reference_model(batch["chosen"]), batch["chosen"], batch["chosen_mask"])
        ref_rejected_log_probas = compute_logprobs(reference_model(batch["rejected"]), batch["rejected"], batch["rejected_mask"])
        ### END YOUR CODE ###
    
    ### START YOUR CODE ###
    # Hint: Call `compute_dpo_loss`
    loss, chosen_rewards, rejected_rewards = compute_dpo_loss(policy_chosen_log_probas, policy_rejected_log_probas, ref_chosen_log_probas, ref_rejected_log_probas, beta)
    ### END YOUR CODE ###

    return loss, chosen_rewards, rejected_rewards

In order to test the function, we need to load two models first.

In [29]:
BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "124M": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "355M": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "774M": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "1558M": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
CHOOSE_MODEL = "355M"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

policy_model = GPTModel(BASE_CONFIG)
reference_model = GPTModel(BASE_CONFIG)

In [30]:
policy_model.load_state_dict(
    torch.load(
        "gpt2-355M-sft.pth", # This should be the path to the SFT tuned model weights
        map_location=torch.device("cpu"),
        weights_only=True
    )
) # policy_model is not in eval() mode

reference_model.load_state_dict(
    torch.load(
        "gpt2-355M-sft.pth", # This should be the path to the SFT tuned model weights
        map_location=torch.device("cpu"),
        weights_only=True
    )
)
policy_model.to(device)
reference_model.to(device)
reference_model.eval();

In [31]:
# Test
with torch.no_grad():
    loss = compute_dpo_loss_batch(batch, policy_model, reference_model, beta=0.1)
print(loss)

# You are expected to see the following output:
# (tensor(0.6541), tensor([0., 0.]), tensor([0., 0.]))

(tensor(0.6931, device='cuda:0'), tensor([0., 0.], device='cuda:0'), tensor([0., 0.], device='cuda:0'))
